In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
-- Analisando alguns dados com SQL
select * from proj_queimadas.bronze.bronze_queimadas
where numero_dias_sem_chuva is  not null


#### Lendo a tabela bronze_queimadas na camada `bronze`

In [0]:

tb_queimadas = (
    spark.table('proj_queimadas.bronze.bronze_queimadas')
)

#### Substituindo valores nas colunas `numero_dias_sem_chuva`,`precipitacao`,`risco_fogo` e `frp` de "-999.0" para "0.0"

In [0]:
tb_queimadas = tb_queimadas.withColumns(
    {
        'numero_dias_sem_chuva':when(col('numero_dias_sem_chuva').startswith('-'),None).otherwise(col('numero_dias_sem_chuva')),
        'precipitacao':when(col('precipitacao').startswith('-'),None).otherwise(col('precipitacao')),
        'risco_fogo':when(col('risco_fogo').startswith('-'),None).otherwise(col('risco_fogo')),
        'frp':when(col('frp').startswith('-'),None).otherwise(col('frp'))
    }
)

#### Colocando colunas no padrão primeira letra em maiúscula

In [0]:
tb_queimadas = tb_queimadas.withColumns(
    {
        'estado':initcap(col('estado')),
        'municipio':initcap(col('municipio'))
    }
)

#### Dividindo a coluna `data_pas` em duas colunas no tipo `DATA` e `HORA`

In [0]:
tb_queimadas =  (tb_queimadas
    .withColumns(
        {
            'data_pas':col('data_pas').cast('date'),
            'hora_pas':substring(col('data_pas'),12,19)
        }
    )
)

#### Tirando espaços em branco nas colunas `latitude` e `longitude`

In [0]:

tb_queimadas =(
    tb_queimadas
    .withColumns(
        {
            'latitude':trim('latitude'),
            'longitude':trim('longitude')
        }
    )
)

#### Tipando as colunas do tb_queimadas

In [0]:
tb_queimadas = (
    tb_queimadas
    .withColumns(
        {
            'latitude':col('latitude').cast('double'),
            'longitude':col('longitude').cast('double'),
            'data_pas':col('data_pas').cast('date'),
            'satelite':col('satelite').cast('string'),
            'pais':col('pais').cast('string'),
            'estado':col('estado').cast('string'),
            'municipio':col('municipio').cast('string'),
            'bioma':col('bioma').cast('string'),
            'numero_dias_sem_chuva':col('numero_dias_sem_chuva').cast('double').cast('int'),
            'precipitacao':col('precipitacao').cast('double'),
            'risco_fogo':col('risco_fogo').cast('double'),
            'id_area_industrial':col('id_area_industrial').cast('int'),
            'frp':col('frp').cast('double'),
            'hora_pas':col('hora_pas').cast('string')
        }
    )
)

#### Salvando tabela tb_queimadas na camada Silver

In [0]:
## tb_queimadas.write.mode('overwrite').format('delta').saveAsTable('proj_queimadas.silver.silver_queimadas')

In [0]:
(
    tb_queimadas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable('proj_queimadas.silver.silver_queimadas')
)